In [7]:
import pandas as pd


data = pd.read_parquet('/dev/shm/ye/rl-data/math-rlvr-unified-dup.math_difficulty.parquet')

rating_pattern = r'["\']?difficult_rating["\']?\s*[:=]\s*["\']?([^,\"\'\}\]\s]+)'
category_pattern = r'["\']?catories["\']?\s*[:=]\s*["\']?([^,\"\'\}\]\n]+)'

sglang_text = data['sglang_result'].fillna('')

data['difficult_rating'] = sglang_text.str.extract(rating_pattern, expand=False).fillna('[invalid]')
data['catories'] = sglang_text.str.extract(category_pattern, expand=False).fillna('[invalid]')

data[['difficult_rating', 'catories']].head()

,difficult_rating,catories
0,2,几何
1,1,应用数学建模
2,8,组合数学
3,4,应用数学建模
4,[invalid],[invalid]


In [10]:
valid_ratings = pd.to_numeric(data['difficult_rating'], errors='coerce')
valid_ratings = valid_ratings[(valid_ratings >= 1) & (valid_ratings <= 10)].astype(int)

rating_distribution = (
    valid_ratings
    .value_counts()
    .sort_index()
    .rename_axis('difficult_rating')
    .reset_index(name='count')
)

rating_distribution['ratio'] = rating_distribution['count'] / rating_distribution['count'].sum()

rating_distribution

,difficult_rating,count,ratio
0,1,24339,0.044795
1,2,100696,0.185329
2,3,80751,0.148620
3,4,80339,0.147862
4,5,66800,0.122944
5,6,86190,0.158631
6,7,60393,0.111152
7,8,31601,0.058161
8,9,12048,0.022174
9,10,180,0.000331


In [12]:
valid_categories = data['catories'].fillna('').astype(str).str.strip()
valid_categories = valid_categories[
    (valid_categories != '') &
    (valid_categories != '[invalid]')
]

category_distribution = (
    valid_categories
    .value_counts()
    .rename_axis('catories')
    .reset_index(name='count')
)

category_distribution['ratio'] = category_distribution['count'] / category_distribution['count'].sum()

category_distribution.head(20)

,catories,count,ratio
0,代数,201707,0.371129
1,几何,94491,0.173858
2,数论,61970,0.114021
3,组合数学,55108,0.101395
4,微积分,52686,0.096939
5,应用数学建模,34281,0.063075
6,概率统计,22935,0.042199
7,线性代数,8898,0.016372
8,其他,5775,0.010626
9,离散数学与逻辑,3770,0.006937


In [14]:
output_path = '/dev/shm/ye/rl-data/math-rlvr-unified-dup.math_difficulty.gt7.parquet'

valid_ratings = pd.to_numeric(data['difficult_rating'], errors='coerce')
filtered_data = data.loc[valid_ratings > 7].copy()
filtered_data['difficult_rating'] = valid_ratings.loc[filtered_data.index].astype(int)

filtered_data.to_parquet(output_path, index=False)

print(f'Saved {len(filtered_data)} rows to {output_path}')
filtered_data.head()

Saved 43829 rows to /dev/shm/ye/rl-data/math-rlvr-unified-dup.math_difficulty.gt7.parquet


,data_source,prompt,ability,reward_model,extra_info,sglang_result,difficult_rating,catories
2,AceReason-Math,[{'content': '58 balls of two colors - red and...,math,"{'ground_truth': '20', 'style': 'rule'}","{'dataset_name': 'AceReason-Math', 'index': 2}",推理：这道题目属于组合数学中的计数与极值问题，具体涉及循环排列、局部模式统计以及奇偶性分析。...,8,组合数学
15,AceReason-Math,"[{'content': 'Consider a sequence $x_1,x_2,\c...",math,"{'ground_truth': '622', 'style': 'rule'}","{'dataset_name': 'AceReason-Math', 'index': 15}",推理：这道题目属于数列与代数领域的综合题，具体涉及非线性递推数列的性质分析、代数变形技巧以及...,8,代数
38,AceReason-Math,[{'content': 'Vasya has selected 8 squares on ...,math,"{'ground_truth': '2', 'style': 'rule'}","{'dataset_name': 'AceReason-Math', 'index': 38}",推理：这道题目属于组合数学与博弈论的交叉领域，具体涉及排列组合、奇偶性分析以及极值策略。\n...,8,组合数学
59,AceReason-Math,[{'content': 'We write on the board the equati...,math,"{'ground_truth': '2016', 'style': 'rule'}","{'dataset_name': 'AceReason-Math', 'index': 59}",推理：这道题目属于组合数学与多项式理论的交叉领域，具体涉及多项式的根分布、因式分解结构以及奇...,8,组合数学
69,AceReason-Math,[{'content': 'An airline company is planning t...,math,"{'ground_truth': '512', 'style': 'rule'}","{'dataset_name': 'AceReason-Math', 'index': 69}",推理：这道题目属于组合数学中的图论计数问题，具体涉及偏序集（Poset）上的子图计数。\n\...,8,组合数学
